In [ ]:
import subprocess, sys
sys.modules['peft'] = None

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "ezdxf[draw]", "easyocr", "diffusers", "transformers",
    "accelerate", "torch", "torchvision",
    "flask", "pyngrok", "Pillow", "opencv-python-headless",
], check=False, capture_output=True)

print("All dependencies installed.")


In [ ]:
# Cell 1: run_pipeline() and helpers
# ─── Imports ──────────────────────────────────────────────────────────────────
import io, base64, os, math
import torch
import ezdxf
from ezdxf.addons.drawing import RenderContext, Frontend
from ezdxf.addons.drawing.matplotlib import MatplotlibBackend
import easyocr
import cv2
import numpy as np
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from diffusers import (
    StableDiffusionControlNetPipeline,
    ControlNetModel,
    UniPCMultistepScheduler,
)

# ─── Room data tables ─────────────────────────────────────────────────────────
ROOM_TYPES = {
    'kitchen': 'kitchen', 'مطبخ': 'kitchen',
    'living': 'living_room', 'معيشة': 'living_room', 'صالة': 'living_room',
    'reception': 'reception', 'استقبال': 'reception', 'salon': 'reception',
    'master': 'bedroom', 'bedroom': 'bedroom', 'غرفة': 'bedroom', 'نوم': 'bedroom',
    'bathroom': 'bathroom', 'حمام': 'bathroom', 'bath': 'bathroom',
    'toilet': 'bathroom', 'wc': 'bathroom',
    'dressing': 'dressing', 'ملابس': 'dressing',
    'dining': 'dining_room', 'سفرة': 'dining_room',
    'office': 'office', 'مكتب': 'office',
    'foyer': 'reception', 'corridor': 'reception', 'ممر': 'reception',
    'store': 'dressing', 'مخزن': 'dressing',
}
STANDARD = {
    'kitchen':     {'w': 4.5, 'd': 3.2, 'windows': 2, 'doors': 1},
    'living_room': {'w': 5.5, 'd': 4.0, 'windows': 3, 'doors': 2},
    'reception':   {'w': 5.0, 'd': 4.5, 'windows': 2, 'doors': 2},
    'bedroom':     {'w': 4.0, 'd': 3.5, 'windows': 2, 'doors': 1},
    'bathroom':    {'w': 2.5, 'd': 2.0, 'windows': 1, 'doors': 1},
    'dressing':    {'w': 2.5, 'd': 2.0, 'windows': 0, 'doors': 1},
    'dining_room': {'w': 4.0, 'd': 3.5, 'windows': 2, 'doors': 1},
    'office':      {'w': 3.5, 'd': 3.0, 'windows': 1, 'doors': 1},
}
STD_FURNITURE = {
    'kitchen':     [{"name":"Counter","w":2.5,"d":0.6,"h":0.9},{"name":"Fridge","w":0.8,"d":0.7,"h":1.8}],
    'living_room': [{"name":"Sofa","w":2.8,"d":0.9,"h":0.8},{"name":"Coffee Table","w":1.2,"d":0.6,"h":0.45}],
    'reception':   [{"name":"Sofa Set","w":3.0,"d":0.9,"h":0.8},{"name":"TV Unit","w":2.0,"d":0.4,"h":0.5}],
    'bedroom':     [{"name":"Bed","w":2.0,"d":1.8,"h":0.5},{"name":"Wardrobe","w":2.0,"d":0.6,"h":2.2}],
    'bathroom':    [{"name":"Shower","w":0.9,"d":0.9,"h":2.0},{"name":"Vanity","w":0.8,"d":0.5,"h":0.85}],
    'dressing':    [{"name":"Wardrobe","w":2.0,"d":0.6,"h":2.2}],
    'dining_room': [{"name":"Dining Table","w":1.8,"d":0.9,"h":0.75}],
    'office':      [{"name":"Desk","w":1.6,"d":0.8,"h":0.75}],
}
PRICES = {
    'kitchen': 6500, 'bathroom': 5500, 'living_room': 4000, 'reception': 4500,
    'bedroom': 3500, 'dressing': 3000, 'dining_room': 4000, 'office': 3500,
}
GENERIC_TYPES = ['living_room', 'bedroom', 'kitchen', 'bathroom', 'bedroom', 'dressing']

# ─── Helpers ──────────────────────────────────────────────────────────────────
def pil_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

def render_dxf_to_pil(dxf_path):
    doc = ezdxf.readfile(dxf_path)
    fig = plt.figure(figsize=(20, 16), dpi=150)
    ax = fig.add_axes([0, 0, 1, 1])
    ctx = RenderContext(doc)
    out = MatplotlibBackend(ax)
    Frontend(ctx, out).draw_layout(doc.modelspace(), finalize=True)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

def extract_dxf_text_labels(dxf_path):
    labels = []
    try:
        doc = ezdxf.readfile(dxf_path)
        msp = doc.modelspace()
        for e in msp:
            try:
                if e.dxftype() == 'TEXT':
                    txt = e.dxf.text.strip()
                    pos = e.dxf.insert
                    if txt and len(txt) >= 2:
                        labels.append({'text': txt, 'x': float(pos.x), 'y': float(pos.y)})
                elif e.dxftype() == 'MTEXT':
                    txt = e.plain_mtext().strip()
                    pos = e.dxf.insert
                    if txt and len(txt) >= 2:
                        labels.append({'text': txt, 'x': float(pos.x), 'y': float(pos.y)})
            except Exception:
                pass
    except Exception as ex:
        print(f"  DXF label warning: {ex}")
    return labels

def classify_room(name):
    name_lower = name.lower()
    for key, val in ROOM_TYPES.items():
        if key in name_lower:
            return val
    return None

def make_room_entry(name, rtype, idx, scale=1.0):
    std = STANDARD.get(rtype, {'w': 3.5, 'd': 3.0, 'windows': 1, 'doors': 1})
    w = round(std['w'] * scale, 2)
    d = round(std['d'] * scale, 2)
    return {
        'id':    f"{rtype}_{idx}",
        'name':  name,
        'type':  rtype,
        'width': w, 'depth': d,
        'area':  round(w * d, 1),
        'windows': std['windows'], 'doors': std['doors'],
        'furniture': [
            {'name': f['name'],
             'w': round(f['w'] * scale, 2),
             'd': round(f['d'] * scale, 2),
             'h': f['h']}
            for f in STD_FURNITURE.get(rtype, [])
        ],
        'price_per_m2':    PRICES.get(rtype, 4000),
        'price_finishing': int(PRICES.get(rtype, 4000) * round(w * d, 1)),
    }

def crop_from_ocr_bbox(floor_np, ocr_bbox, expand=4.0):
    """
    Expand an OCR bounding box by `expand` factor to get a room-sized crop.
    ocr_bbox is a list of 4 [x,y] corner points (from easyocr).
    """
    h, w = floor_np.shape[:2]
    xs = [p[0] for p in ocr_bbox]
    ys = [p[1] for p in ocr_bbox]
    cx = sum(xs) / 4
    cy = sum(ys) / 4
    bw = max(xs) - min(xs)
    bh = max(ys) - min(ys)
    half_w = max(bw * expand, w * 0.12) / 2
    half_h = max(bh * expand, h * 0.12) / 2
    x1 = int(max(0, cx - half_w))
    x2 = int(min(w, cx + half_w))
    y1 = int(max(0, cy - half_h))
    y2 = int(min(h, cy + half_h))
    if x2 <= x1 or y2 <= y1:
        return None
    return floor_np[y1:y2, x1:x2]

def grid_crops(floor_np, n):
    """Divide the floor plan image into n roughly equal tiles."""
    h, w = floor_np.shape[:2]
    cols = math.ceil(math.sqrt(n))
    rows = math.ceil(n / cols)
    tw = w // cols
    th = h // rows
    crops = []
    for r in range(rows):
        for c in range(cols):
            if len(crops) >= n:
                break
            x1, y1 = c * tw, r * th
            x2, y2 = min(w, x1 + tw), min(h, y1 + th)
            crops.append(floor_np[y1:y2, x1:x2])
    return crops

# ─── Main pipeline ────────────────────────────────────────────────────────────
def run_pipeline(dxf_path, area_m2, style, palette, pipe_model, ocr_model):
    # Step 1: DXF text entities
    print("[1/5] Extracting labels from DXF entities...")
    dxf_labels = extract_dxf_text_labels(dxf_path)
    print(f"  Found {len(dxf_labels)} text entities in DXF")

    # Step 2: render DXF to PNG
    print("[2/5] Rendering DXF to image...")
    floor_pil = render_dxf_to_pil(dxf_path)
    floor_np  = np.array(floor_pil)
    h_img, w_img = floor_np.shape[:2]
    img_path  = dxf_path.replace('.dxf', '_render.png')
    floor_pil.save(img_path)
    print(f"  Image size: {w_img}x{h_img}")

    # Step 3: OCR
    print("[3/5] Running OCR (en + ar)...")
    try:
        ocr_results = ocr_model.readtext(img_path)
    except Exception as ex:
        print(f"  OCR error: {ex}")
        ocr_results = []
    print(f"  OCR found {len(ocr_results)} regions")

    # Step 4: build room list — collect OCR bboxes for matched rooms
    print("[4/5] Detecting and scaling rooms...")
    skip = {'tv', 'ac', 'door', 'window', 'elev', 'stair', 'shaft', 'duct'}
    final_rooms = []
    seen = set()

    # Priority 1: OCR (has pixel bounding boxes we can use for cropping)
    for result in ocr_results:
        bbox, text, conf = result
        text = text.strip()
        if conf < 0.25 or len(text) < 2:
            continue
        name = text.upper().replace(':', '').replace('[', '').replace(']', '').strip()
        if name in seen or any(s in name.lower() for s in skip):
            continue
        clean = name.replace('.', '').replace('M', '').replace('X', '').replace(' ', '').replace('2', '')
        if all(c.isdigit() for c in clean) or not clean:
            continue
        rtype = classify_room(name)
        if rtype:
            seen.add(name)
            final_rooms.append({
                'name': name,
                'type': rtype,
                'area_standard': STANDARD[rtype]['w'] * STANDARD[rtype]['d'],
                'ocr_bbox': bbox,   # pixel bounding box for cropping!
            })

    # Priority 2: DXF text labels that OCR missed (no bbox — will use grid tile)
    for lbl in dxf_labels:
        name = lbl['text'].strip()
        if not name or name in seen or any(s in name.lower() for s in skip):
            continue
        rtype = classify_room(name)
        if rtype:
            seen.add(name)
            final_rooms.append({
                'name': name.upper(),
                'type': rtype,
                'area_standard': STANDARD[rtype]['w'] * STANDARD[rtype]['d'],
                'ocr_bbox': None,   # no pixel position — will fall back to grid tile
            })

    print(f"  {len(final_rooms)} labelled rooms found")

    # Fallback: if still 0 rooms, create generic rooms from a grid
    if not final_rooms:
        print("  No labels at all — using 4-tile grid fallback")
        for i in range(4):
            rtype = GENERIC_TYPES[i % len(GENERIC_TYPES)]
            final_rooms.append({
                'name': f"{rtype.replace('_', ' ').title()} {i+1}",
                'type': rtype,
                'area_standard': STANDARD[rtype]['w'] * STANDARD[rtype]['d'],
                'ocr_bbox': None,
            })

    # Compute scale factor
    total_std = sum(r['area_standard'] for r in final_rooms)
    scale = (area_m2 / total_std) ** 0.5 if total_std > 0 else 1.0
    print(f"  scale={scale:.3f}")

    # Prepare grid tiles as fallback crops for rooms without OCR bbox
    grid_tiles = grid_crops(floor_np, len(final_rooms))

    # Step 5: for each room — get crop, run Canny + SD
    print("[5/5] Generating interior designs...")
    neg = (
        "deformed, distorted, blurry, low quality, cartoon, text, watermark, "
        "top-down view, floor plan, blueprint, ceiling view"
    )
    results_out = []

    for idx, room in enumerate(final_rooms):
        # Get crop: prefer OCR bbox expansion, else grid tile
        raw_crop = None
        if room['ocr_bbox'] is not None:
            raw_crop = crop_from_ocr_bbox(floor_np, room['ocr_bbox'], expand=4.0)

        if raw_crop is None or raw_crop.size == 0:
            # Use corresponding grid tile
            raw_crop = grid_tiles[idx] if idx < len(grid_tiles) else floor_np

        crop_pil = Image.fromarray(raw_crop).resize((512, 512))

        gray_c = cv2.cvtColor(np.array(crop_pil), cv2.COLOR_RGB2GRAY)
        edges  = cv2.Canny(gray_c, 50, 150)
        edges_pil = Image.fromarray(edges).convert("RGB")

        rtype_str = room['type'].replace('_', ' ')
        entry = make_room_entry(room['name'], room['type'], idx, scale)

        prompt = (
            f"RAW photo, photorealistic {rtype_str} interior, "
            f"{style} style, {palette} color palette, "
            "modern furniture, natural daylight, architectural visualization, "
            "8k uhd, high quality, perspective view from inside the room"
        )

        print(f"  [{idx+1}/{len(final_rooms)}] {room['name']}...", end='', flush=True)
        result_img = pipe_model(
            prompt=prompt,
            negative_prompt=neg,
            image=edges_pil,
            num_inference_steps=25,
            guidance_scale=7.0,
            controlnet_conditioning_scale=0.6,
            generator=torch.Generator("cuda").manual_seed(42 + idx),
        ).images[0]
        print(" done")

        results_out.append({
            **entry,
            'crop_image_b64':      pil_to_b64(crop_pil),
            'canny_image_b64':     pil_to_b64(edges_pil),
            'generated_image_b64': pil_to_b64(result_img),
        })

    print(f"\nPipeline complete -- {len(results_out)} room(s) processed.")
    return results_out

print("run_pipeline() ready. Proceed to Cell 2 to load models.")


In [ ]:
# ─── Load AI models once (takes ~3 min on first run) ─────────────────────────
from flask import Flask, request, jsonify
import tempfile

print("Loading ControlNet (lllyasviel/sd-controlnet-canny)...")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch.float16,
)
print("Loading Realistic Vision V5.1...")
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "SG161222/Realistic_Vision_V5.1_noVAE",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to("cuda")
pipe.enable_attention_slicing()

print("Loading EasyOCR (en)...")
ocr_reader = easyocr.Reader(['en', 'ar'], gpu=torch.cuda.is_available())

print("All models ready!")

# ─── Flask application ────────────────────────────────────────────────────────
app = Flask(__name__)

@app.route("/health")
def health():
    return jsonify({"status": "ok", "gpu": torch.cuda.is_available()})

@app.route("/process", methods=["POST"])
def process():
    dxf_file = request.files.get("dxf_file")
    if not dxf_file:
        return jsonify({"error": "dxf_file is required"}), 400

    area_m2 = float(request.form.get("area_m2", 100))
    style   = request.form.get("style",   "modern")
    palette = request.form.get("palette", "neutral")

    with tempfile.NamedTemporaryFile(suffix=".dxf", delete=False) as tmp:
        dxf_file.save(tmp.name)
        tmp_path = tmp.name

    try:
        rooms = run_pipeline(tmp_path, area_m2, style, palette, pipe, ocr_reader)
        return jsonify({"rooms": rooms, "total_rooms": len(rooms)})
    except Exception as e:
        import traceback
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500
    finally:
        try:
            os.unlink(tmp_path)
        except Exception:
            pass

print("Flask app defined. Run the next cell to start the server.")


In [ ]:
# ─── Start ngrok + Flask ──────────────────────────────────────────────────────
# 1. Get your free auth token: https://dashboard.ngrok.com/get-started/your-authtoken
# 2. Paste it below, then run this cell.
# 3. Copy the printed CAD_PIPELINE_URL= line into backend/.env and restart the backend.

from pyngrok import ngrok, conf

NGROK_TOKEN = "3DSP6fOiQUEOfSlHFzc7Y7Frgzk_wNQArDfhfbn12AVoSmB6"   # <── paste your token here
conf.get_default().auth_token = NGROK_TOKEN

tunnel = ngrok.connect(5000)
public_url = tunnel.public_url

print()
print("=" * 55)
print("  KAGGLE ENDPOINT IS LIVE")
print("=" * 55)
print(f"  Paste into backend/.env:")
print()
print(f"    CAD_PIPELINE_URL={public_url}")
print()
print(f"  Health check: {public_url}/health")
print("=" * 55)
print()

# Flask runs in the foreground — keep this cell running
app.run(host="0.0.0.0", port=5000)


In [ ]:
# =======================================
# Keep-Alive System للتجربة
# =======================================

import threading
import time
import subprocess
from IPython.display import clear_output
from datetime import datetime

# Global control
keep_alive_active = False
start_time = None
alive_thread = None

def keep_alive_worker():
    """الـ function اللي هتشتغل في background"""
    global keep_alive_active, start_time
    counter = 0
    
    while keep_alive_active:
        counter += 1
        current_time = datetime.now()
        elapsed = time.time() - start_time
        hours = int(elapsed // 3600)
        minutes = int((elapsed % 3600) // 60)
        
        clear_output(wait=True)
        print(f"🟢 Keep-Alive Status")
        print(f"═" * 40)
        print(f"📊 Runtime: {hours:02d}h {minutes:02d}m")
        print(f"🔄 Heartbeat: #{counter}")
        print(f"🕐 Current time: {current_time.strftime('%H:%M:%S')}")
        print(f"⚠️  Normal Kaggle timeout: 30 minutes")
        
        # Check GPU status
        try:
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=name,utilization.gpu', '--format=csv,noheader'],
                capture_output=True, text=True, timeout=5
            )
            if result.returncode == 0:
                gpu_info = result.stdout.strip()
                print(f"🎮 GPU: ✅ Available - {gpu_info}")
            else:
                print(f"🎮 GPU: ❌ Check failed")
        except:
            print(f"🎮 GPU: ❌ Lost or error")
        
        print(f"\n💡 To stop: run 'stop_keep_alive()'")
        print(f"═" * 40)
        
        # Sleep for 5 minutes
        time.sleep(300)
    
    print("🔴 Keep-alive stopped")

def start_keep_alive():
    """تشغيل الـ keep-alive"""
    global keep_alive_active, start_time, alive_thread
    
    if keep_alive_active:
        print("⚠️  Keep-alive already running!")
        return
    
    keep_alive_active = True
    start_time = time.time()
    
    alive_thread = threading.Thread(target=keep_alive_worker, daemon=True)
    alive_thread.start()
    
    print("✅ Keep-alive started!")
    print("📝 Will update every 5 minutes")
    print("🛑 To stop: run stop_keep_alive()")

def stop_keep_alive():
    """إيقاف الـ keep-alive"""
    global keep_alive_active
    keep_alive_active = False
    print("🔴 Keep-alive stopping...")

def status_keep_alive():
    """حالة الـ keep-alive"""
    global keep_alive_active, start_time
    
    if keep_alive_active and start_time:
        elapsed = time.time() - start_time
        hours = int(elapsed // 3600)
        minutes = int((elapsed % 3600) // 60)
        print(f"🟢 Keep-alive active - Runtime: {hours:02d}h {minutes:02d}m")
    else:
        print("🔴 Keep-alive not active")

# عرض الأوامر المتاحة
print("🎮 Keep-Alive Commands:")
print("═" * 30)
print("start_keep_alive()    - تشغيل")
print("stop_keep_alive()     - إيقاف") 
print("status_keep_alive()   - الحالة")
print("═" * 30)
print("\n💡 للتجربة: شغّل start_keep_alive() وراقب لمدة ساعة")

In [ ]:
start_keep_alive()

In [ ]:
status_keep_alive()

In [ ]:
stop_keep_alive()